# Kvasir-SEG Split and Integrity Validation

Validates metadata and split integrity from `0_dataset_prep/out/metadata/metadata_enriched.csv`.\n\nOutputs:\n- `out/metadata/integrity_report.json`\n- `out/metadata/split_summary.csv`

In [1]:
import sys
from pathlib import Path

def _bootstrap_kvasir_seg_path() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / 'utils' / 'segmentation_common.py').exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p
        alt = p / 'Prototyping_reformat' / 'DatasetAnalysis' / 'Kvasir_SEG'
        if (alt / 'utils' / 'segmentation_common.py').exists():
            if str(alt) not in sys.path:
                sys.path.insert(0, str(alt))
            return alt
    raise RuntimeError('Could not locate Kvasir_SEG utils path from current working directory.')

BOOTSTRAP_ROOT = _bootstrap_kvasir_seg_path()
print('BOOTSTRAP_ROOT:', BOOTSTRAP_ROOT)

BOOTSTRAP_ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG


In [2]:

import json
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

from utils.segmentation_common import find_kvasir_seg_root, load_metadata

ROOT = find_kvasir_seg_root()
META_CSV = ROOT / '0_dataset_prep' / 'out' / 'metadata' / 'metadata_enriched.csv'
OUT_META_DIR = ROOT / '0_dataset_prep' / 'out' / 'metadata'
OUT_META_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('META_CSV:', META_CSV)


ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG
META_CSV: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/0_dataset_prep/out/metadata/metadata_enriched.csv


In [3]:

df = load_metadata(META_CSV)

report = {
    'n_rows': int(len(df)),
    'n_unique_img_id': int(df['img_id'].nunique()),
    'n_unique_image_path': int(df['image_path'].nunique()),
    'n_unique_mask_path': int(df['mask_path'].nunique()),
}

# Basic file existence checks
img_exists = df['image_path'].map(lambda p: Path(p).exists())
mask_exists = df['mask_path'].map(lambda p: Path(p).exists())
report['missing_images'] = int((~img_exists).sum())
report['missing_masks'] = int((~mask_exists).sum())

# Duplicate checks
report['duplicate_img_id'] = int(df['img_id'].duplicated().sum())
report['duplicate_image_path'] = int(df['image_path'].duplicated().sum())
report['duplicate_mask_path'] = int(df['mask_path'].duplicated().sum())

# Split leakage checks
split_sets = {
    s: set(df.loc[df['split'] == s, 'img_id'].astype(str).tolist())
    for s in ['train', 'val', 'test']
}
report['leak_train_val'] = int(len(split_sets['train'] & split_sets['val']))
report['leak_train_test'] = int(len(split_sets['train'] & split_sets['test']))
report['leak_val_test'] = int(len(split_sets['val'] & split_sets['test']))

# Split balance
split_summary = (
    df.groupby('split')
      .agg(
          n=('img_id', 'count'),
          mean_width=('width', 'mean'),
          mean_height=('height', 'mean'),
          mean_mask_area_ratio=('mask_area_ratio', 'mean'),
          mean_component_count=('component_count', 'mean'),
          mean_bbox_count=('bbox_count', 'mean'),
      )
      .reset_index()
)


In [4]:

# Validate masks are not empty/corrupt and estimate binary quality
sample_mask_issues = []
empty_masks = 0
non_binary_ratio = 0

for row in df.itertuples(index=False):
    m = np.asarray(Image.open(row.mask_path).convert('L'))
    if m.size == 0:
        sample_mask_issues.append({'img_id': row.img_id, 'issue': 'empty_array'})
        continue

    fg = (m > 127).sum()
    if fg == 0:
        empty_masks += 1

    uniq = np.unique(m)
    if len(uniq) > 16:  # compressed jpg masks are not strictly binary; this is only a soft signal
        non_binary_ratio += 1

report['empty_masks_after_threshold'] = int(empty_masks)
report['high_graylevel_var_masks'] = int(non_binary_ratio)
report['sample_mask_issues'] = sample_mask_issues[:10]


In [5]:

# Save artifacts
integrity_path = OUT_META_DIR / 'integrity_report.json'
split_summary_path = OUT_META_DIR / 'split_summary.csv'

with open(integrity_path, 'w') as f:
    json.dump(report, f, indent=2)

split_summary.to_csv(split_summary_path, index=False)

print('Saved:', integrity_path)
print('Saved:', split_summary_path)
print(json.dumps(report, indent=2))
display(split_summary)


Saved: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/0_dataset_prep/out/metadata/integrity_report.json
Saved: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/0_dataset_prep/out/metadata/split_summary.csv
{
  "n_rows": 1000,
  "n_unique_img_id": 1000,
  "n_unique_image_path": 1000,
  "n_unique_mask_path": 1000,
  "missing_images": 0,
  "missing_masks": 0,
  "duplicate_img_id": 0,
  "duplicate_image_path": 0,
  "duplicate_mask_path": 0,
  "leak_train_val": 0,
  "leak_train_test": 0,
  "leak_val_test": 0,
  "empty_masks_after_threshold": 0,
  "high_graylevel_var_masks": 117,
  "sample_mask_issues": []
}


,split,n,mean_width,mean_height,mean_mask_area_ratio,mean_component_count,mean_bbox_count
0,test,100,621.60000,546.49000,0.161816,1.19000,1.0600
1,train,800,625.24875,544.88375,0.151866,1.19875,1.0625
2,val,100,629.33000,546.72000,0.162356,1.29000,1.1500
